# Zalo AI Challenge 2026 - DataMate Track
## Hybrid SLM (Qwen2.5-7B / 14B) + Deterministic Business Engine

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab (GPU T4 / A100 / L4)**:
1. Cài đặt môi trường siêu tốc.
2. Sử dụng mô hình SLM hàng đầu thế giới phân khúc `<= 15B`: **`Qwen/Qwen2.5-7B-Instruct`** (hoặc 14B).
3. Tích hợp động cơ tính toán Pandas tuân thủ 100% Rulebook §1-10 của BTC.
4. Tự động sinh file `submission.jsonl` đạt chuẩn để nộp bài.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q pandas openpyxl transformers accelerate bitsandbytes torch torchvision

### Bước 2: Tải dữ liệu thi đấu (Upload hoặc kết nối Google Drive)
Bạn có thể nén thư mục `bài làm nộp thử` và `DATA MATE(which we will choose)` thành file zip rồi tải lên Colab, hoặc mount Drive.

In [ ]:
# Nếu bạn lưu dữ liệu trên Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

import os
import json
import pandas as pd
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

### Bước 3: Tải mô hình SLM (Qwen2.5-7B-Instruct 4-bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Dang tai mo hinh {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Mo hinh da san sang!")

### Bước 4: Chạy Pipeline suy luận & sinh kết quả nộp bài

In [ ]:
# Chay pipeline suy luan truc tiep:
!python run_local.py --split public_test --use_slm --slm_provider transformers --output submission_colab.jsonl

# Kiem tra file ket qua:
!head -n 5 submission_colab.jsonl
!wc -l submission_colab.jsonl

### Bước 5: Tải file `submission_colab.jsonl` về máy để nộp lên cổng Zalo AI

In [ ]:
from google.colab import files
files.download('submission_colab.jsonl')